# Phase 4: lighting-robust model in Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhavilkawdia5-ux/leaf_disease_detection/blob/main/ml/notebooks/phase4_robust.ipynb)

This runs Phase 4 of `docs/SYSTEM_DESIGN.md` on this Colab machine. It:

1. downloads and merges the public datasets, as in Phase 2 and 3,
2. trains three models with random lighting augmentation (`ml/src/augment.py`):
   no lighting fix, grey-world colour balance, and CLAHE (`ml/src/preprocess.py`),
3. tests each per lighting condition against `ml/reports/baseline.md` and chooses its
   "not sure" threshold from the val split,
4. picks the best model by **val** accuracy (never by test) and exports it to
   TensorFlow Lite (float16) with size, answer and speed checks,
5. lets you download the reports and the `.tflite`.

**Use a GPU:** Runtime > Change runtime type > T4 GPU. About 30 minutes of downloads,
then roughly 25 to 40 minutes per model. Keep the tab open. Run each cell with
**Shift + Enter**, top to bottom.

## 1. Get the code

In [ ]:
import os
if os.path.isdir("/content/leaf_disease_detection"):
    !git -C /content/leaf_disease_detection pull -q
else:
    !git clone -q https://github.com/bhavilkawdia5-ux/leaf_disease_detection.git /content/leaf_disease_detection
%cd /content/leaf_disease_detection
!git log --oneline -1

## 2. Install packages and check the GPU

TensorFlow and OpenCV are already in Colab.

In [ ]:
!pip install -q pyyaml pyarrow pillow
import tensorflow as tf, cv2
print("TensorFlow", tf.__version__, "| OpenCV", cv2.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none - training will be very slow")

## 3. Download and merge the datasets

About 15 to 30 minutes. If a dataset says FAILED, run the cell again. The counts should
match `docs/datasets.md` (18,455 images).

In [ ]:
!python ml/src/download.py --data /content/data
!python ml/src/dataset.py --data /content/data | tee /content/phase4_counts.txt

## 4. Quick check

About 5 minutes on 20 images per class: checks that augmentation, CLAHE, testing and
export all run. Ignore its accuracy.

In [ ]:
!python ml/src/train.py --data /content/data --out /content/models_check/aug_clahe --augment --preprocess clahe --limit 20 --epochs-head 1 --epochs-fine 1
!python ml/src/evaluate.py --data /content/data --models /content/models_check/aug_clahe --report /content/check/check.md --title "Quick check" --baseline ml/reports/baseline.md --limit 20
!python ml/src/export_tflite.py --data /content/data --models /content/models_check/aug_clahe --report /content/check/export.md

## 5. Train and test the three models

One cell per model, about 25 to 40 minutes each. Each writes
`ml/reports/phase4_<name>.md` and a `.json` with its main numbers. If Colab disconnects,
run steps 1 to 3 again, then only the models that are missing.

In [ ]:
EXPERIMENTS = {  # name: (train.py options, report title)
    "aug": ("--augment", "Phase 4: lighting augmentation"),
    "aug_grey_world": ("--augment --preprocess grey_world", "Phase 4: augmentation + grey-world"),
    "aug_clahe": ("--augment --preprocess clahe", "Phase 4: augmentation + CLAHE"),
}

def run(name):
    options, title = EXPERIMENTS[name]
    !python ml/src/train.py --data /content/data --out /content/models/{name} {options}
    !python ml/src/evaluate.py --data /content/data --models /content/models/{name} --report ml/reports/phase4_{name}.md --title "{title}" --baseline ml/reports/baseline.md

In [ ]:
run("aug")

In [ ]:
run("aug_grey_world")

In [ ]:
run("aug_clahe")

## 6. Compare and pick the best model

The best model is the one with the highest **val** accuracy averaged over the four
lighting conditions. Writes `ml/reports/phase4_comparison.md`.

In [ ]:
import glob, json
from IPython.display import Markdown, display
runs_arg = " ".join(sorted(glob.glob("ml/reports/phase4_aug*.json")))
!python ml/src/evaluate.py --compare {runs_arg} --baseline ml/reports/baseline.md --report ml/reports/phase4_comparison.md
best = json.load(open("ml/reports/phase4_comparison.json"))
best_dir = best["models"]
print("Best:", best)
display(Markdown(open("ml/reports/phase4_comparison.md", encoding="utf-8").read()))

## 7. Export the best model to TensorFlow Lite

Float16 weights. Checks the size (under 20 MB), that it gives the same answers as the
Keras model, and its speed on this machine's CPU. Writes `ml/reports/export.md`.

In [ ]:
!python ml/src/export_tflite.py --data /content/data --models {best_dir} --report ml/reports/export.md
display(Markdown(open("ml/reports/export.md", encoding="utf-8").read()))

## 8. Download the results

One zip with every Phase 4 report and the best model (`model.tflite`, `model.keras`,
`labels.txt`, `train_info.json`).

In [ ]:
!rm -f /content/phase4_results.zip
!zip -j -q /content/phase4_results.zip ml/reports/phase4_*.md ml/reports/phase4_*.json ml/reports/export.md
!cd {best_dir} && zip -q /content/phase4_results.zip model.tflite model.keras labels.txt train_info.json
from google.colab import files
files.download("/content/phase4_results.zip")

## 9. What to do next

1. Unzip: put the `.md` and `.json` reports in `ml/reports/` and the model files in
   `ml/models/<best name>/` (git-ignored) on your laptop.
2. Tell Claude Code they are there. Phase 4 is done when shadow and harsh-sun accuracy
   improved, the model is under 20 MB and under 1 second on a budget phone, and the
   "not sure" threshold is recorded.
3. If a cell fails, copy its output and paste it to Claude Code.